In [0]:
%sql

-- ATLAS Marine — Notebook 2B: Silver Species
-- Transforms four Bronze species tracking tables into clean
-- validated Silver tables with H3 geospatial indexing


-- Pre-flight check — confirm all four species Bronze tables exist
SELECT 'bronze_manta_ray_raw' AS table_name, count(*) AS rows FROM mcphersonsharyn_bronze.bronze_manta_ray_raw
UNION ALL
SELECT 'bronze_whale_shark_raw', count(*) FROM mcphersonsharyn_bronze.bronze_whale_shark_raw
UNION ALL
SELECT 'bronze_sperm_whale_raw', count(*) FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw
UNION ALL
SELECT 'bronze_green_turtle_raw', count(*) FROM mcphersonsharyn_bronze.bronze_green_turtle_raw
ORDER BY table_name;
-- Confirms all four species Bronze tables exist before building Silver


In [0]:
%sql
-- ═══════════════════════════════════════════════════════════════
-- SECTION 1 — REEF MANTA RAY SILVER
-- Source: Movebank — Reef manta ray (Mobula alfredi) home range
-- in Indonesia — Raja Ampat study area
-- 25 individual animals — 3,224 observations — 2014 to 2022
-- GPS tracking — high accuracy positioning
-- ═══════════════════════════════════════════════════════════════

DROP TABLE IF EXISTS mcphersonsharyn_silver.manta_ray_silver;

CREATE TABLE mcphersonsharyn_silver.manta_ray_silver
USING DELTA AS
SELECT
  CAST(`event-id` AS BIGINT) AS event_id,
  1 AS species_id,
  'Reef Manta Ray' AS species_common,
  'Mobula alfredi' AS species_latin,
  CAST(`individual-local-identifier` AS STRING) AS animal_id,
  TO_TIMESTAMP(`timestamp`) AS observation_timestamp,
  TO_DATE(`timestamp`) AS observation_date,
  YEAR(TO_TIMESTAMP(`timestamp`)) AS year,
  MONTH(TO_TIMESTAMP(`timestamp`)) AS month,
  CAST(`location-lat` AS DOUBLE) AS latitude,
  CAST(`location-long` AS DOUBLE) AS longitude,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4) AS h3_res4,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5) AS h3_res5,
  locality,
  CAST(`manually-marked-outlier` AS BOOLEAN) AS is_outlier,
  `sensor-type` AS sensor_type,
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')) AS date_id
FROM mcphersonsharyn_bronze.bronze_manta_ray_raw
WHERE `location-lat` IS NOT NULL
AND `location-long` IS NOT NULL
AND CAST(`location-lat` AS DOUBLE) BETWEEN -90 AND 90
AND CAST(`location-long` AS DOUBLE) BETWEEN -180 AND 180
AND (`manually-marked-outlier` IS NULL OR `manually-marked-outlier` = 'false');
-- Filters null coordinates and physically impossible values
-- Removes manually marked outliers
-- Adds H3 index at resolution 4 and 5
-- date_id matches dim_date format for Gold joins
-- species_id = 1 matches dim_species

In [0]:
%sql
SELECT
  'manta_ray_silver' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT animal_id) AS unique_animals,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3,
  count(CASE WHEN is_outlier = TRUE THEN 1 END) AS outliers_removed
FROM mcphersonsharyn_silver.manta_ray_silver;
-- Expected: approximately 3,224 rows
-- 25 unique animals
-- null_h3 should be 0

In [0]:
%sql

-- SECTION 2 — WHALE SHARK SILVER
-- Source: Movebank — Whale shark (Rhincodon typus)
-- Gulf of Mexico study area
-- 41 individual animals — 3,382 observations — 2009 to 2015
-- Argos satellite tracking — lower accuracy than GPS
-- argos:lc = location class — quality indicator


DROP TABLE IF EXISTS mcphersonsharyn_silver.whale_shark_silver;

CREATE TABLE mcphersonsharyn_silver.whale_shark_silver
USING DELTA AS
SELECT
  CAST(`event-id` AS BIGINT) AS event_id,
  2 AS species_id,
  'Whale Shark' AS species_common,
  'Rhincodon typus' AS species_latin,
  CAST(`individual-local-identifier` AS STRING) AS animal_id,
  TO_TIMESTAMP(`timestamp`) AS observation_timestamp,
  TO_DATE(`timestamp`) AS observation_date,
  YEAR(TO_TIMESTAMP(`timestamp`)) AS year,
  MONTH(TO_TIMESTAMP(`timestamp`)) AS month,
  CAST(`location-lat` AS DOUBLE) AS latitude,
  CAST(`location-long` AS DOUBLE) AS longitude,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4) AS h3_res4,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5) AS h3_res5,
  NULL AS locality,
  CAST(`algorithm-marked-outlier` AS BOOLEAN) AS is_outlier,
  `sensor-type` AS sensor_type,
  `argos:lc` AS argos_location_class,
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')) AS date_id
FROM mcphersonsharyn_bronze.bronze_whale_shark_raw
WHERE `location-lat` IS NOT NULL
AND `location-long` IS NOT NULL
AND CAST(`location-lat` AS DOUBLE) BETWEEN -90 AND 90
AND CAST(`location-long` AS DOUBLE) BETWEEN -180 AND 180
AND (`algorithm-marked-outlier` IS NULL OR `algorithm-marked-outlier` = 'false');
-- argos_location_class included — quality indicator for Argos tracking
-- A and B class locations have lower accuracy than GPS
-- algorithm-marked-outlier filter removes flagged positions
-- species_id = 2 matches dim_species

In [0]:
%sql
SELECT
  'whale_shark_silver' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT animal_id) AS unique_animals,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  count(DISTINCT argos_location_class) AS location_classes,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3
FROM mcphersonsharyn_silver.whale_shark_silver;
-- Expected: approximately 3,382 rows
-- 41 unique animals
-- Multiple Argos location classes expected

In [0]:
%sql

-- SECTION 3 — SPERM WHALE SILVER
-- Source: Movebank — Sperm whale (Physeter macrocephalus)
-- Gulf of Mexico study area
-- 17 individual animals — 7,759 observations — 2011 to 2013
-- GPS tracking — high accuracy positioning


DROP TABLE IF EXISTS mcphersonsharyn_silver.sperm_whale_silver;

CREATE TABLE mcphersonsharyn_silver.sperm_whale_silver
USING DELTA AS
SELECT
  CAST(`event-id` AS BIGINT) AS event_id,
  3 AS species_id,
  'Sperm Whale' AS species_common,
  'Physeter macrocephalus' AS species_latin,
  CAST(`individual-local-identifier` AS STRING) AS animal_id,
  TO_TIMESTAMP(`timestamp`) AS observation_timestamp,
  TO_DATE(`timestamp`) AS observation_date,
  YEAR(TO_TIMESTAMP(`timestamp`)) AS year,
  MONTH(TO_TIMESTAMP(`timestamp`)) AS month,
  CAST(`location-lat` AS DOUBLE) AS latitude,
  CAST(`location-long` AS DOUBLE) AS longitude,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4) AS h3_res4,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5) AS h3_res5,
  NULL AS locality,
  CAST(`manually-marked-outlier` AS BOOLEAN) AS is_outlier,
  `sensor-type` AS sensor_type,
  NULL AS argos_location_class,
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')) AS date_id
FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw
WHERE `location-lat` IS NOT NULL
AND `location-long` IS NOT NULL
AND CAST(`location-lat` AS DOUBLE) BETWEEN -90 AND 90
AND CAST(`location-long` AS DOUBLE) BETWEEN -180 AND 180
AND (`manually-marked-outlier` IS NULL OR `manually-marked-outlier` = 'false');
-- GPS tracking — higher accuracy than Argos
-- argos_location_class NULL — not applicable for GPS
-- species_id = 3 matches dim_species

In [0]:
%sql
SELECT
  'sperm_whale_silver' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT animal_id) AS unique_animals,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3
FROM mcphersonsharyn_silver.sperm_whale_silver;
-- Expected: approximately 7,759 rows
-- 17 unique animals
-- null_h3 should be 0

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.green_turtle_silver;

CREATE TABLE mcphersonsharyn_silver.green_turtle_silver
USING DELTA AS
SELECT
  CAST(`event-id` AS BIGINT) AS event_id,
  4 AS species_id,
  'Green Turtle' AS species_common,
  'Chelonia mydas' AS species_latin,
  CAST(`individual-local-identifier` AS STRING) AS animal_id,
  TO_TIMESTAMP(`timestamp`) AS observation_timestamp,
  TO_DATE(`timestamp`) AS observation_date,
  YEAR(TO_TIMESTAMP(`timestamp`)) AS year,
  MONTH(TO_TIMESTAMP(`timestamp`)) AS month,
  CAST(`location-lat` AS DOUBLE) AS latitude,
  CAST(`location-long` AS DOUBLE) AS longitude,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4) AS h3_res4,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5) AS h3_res5,
  NULL AS locality,
  CAST(`algorithm-marked-outlier` AS BOOLEAN) AS is_outlier,
  `sensor-type` AS sensor_type,
  `argos:lc` AS argos_location_class,
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')) AS date_id
FROM mcphersonsharyn_bronze.bronze_green_turtle_raw
WHERE `location-lat` IS NOT NULL
AND `location-long` IS NOT NULL
AND CAST(`location-lat` AS DOUBLE) BETWEEN -90 AND 90
AND CAST(`location-long` AS DOUBLE) BETWEEN -180 AND 180
AND (`algorithm-marked-outlier` IS NULL OR `algorithm-marked-outlier` = 'false');

In [0]:
%sql
SELECT
  'green_turtle_silver' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT animal_id) AS unique_animals,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  count(DISTINCT argos_location_class) AS location_classes,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3
FROM mcphersonsharyn_silver.green_turtle_silver;

In [0]:
%sql
CREATE OR REPLACE VIEW mcphersonsharyn_silver.migration_silver AS
SELECT * FROM mcphersonsharyn_silver.manta_ray_silver
UNION ALL
SELECT * FROM mcphersonsharyn_silver.whale_shark_silver
UNION ALL
SELECT * FROM mcphersonsharyn_silver.sperm_whale_silver
UNION ALL
SELECT * FROM mcphersonsharyn_silver.green_turtle_silver;
-- Unified view across all four species Silver tables
-- CREATE OR REPLACE handles existing view without needing DROP first

In [0]:
%sql
CREATE OR REPLACE VIEW mcphersonsharyn_silver.migration_silver AS
SELECT
  event_id, species_id, species_common, species_latin,
  animal_id, observation_timestamp, observation_date,
  year, month, latitude, longitude,
  h3_res4, h3_res5, locality, is_outlier,
  sensor_type, NULL AS argos_location_class, date_id
FROM mcphersonsharyn_silver.manta_ray_silver

UNION ALL

SELECT
  event_id, species_id, species_common, species_latin,
  animal_id, observation_timestamp, observation_date,
  year, month, latitude, longitude,
  h3_res4, h3_res5, locality, is_outlier,
  sensor_type, argos_location_class, date_id
FROM mcphersonsharyn_silver.whale_shark_silver

UNION ALL

SELECT
  event_id, species_id, species_common, species_latin,
  animal_id, observation_timestamp, observation_date,
  year, month, latitude, longitude,
  h3_res4, h3_res5, locality, is_outlier,
  sensor_type, NULL AS argos_location_class, date_id
FROM mcphersonsharyn_silver.sperm_whale_silver

UNION ALL

SELECT
  event_id, species_id, species_common, species_latin,
  animal_id, observation_timestamp, observation_date,
  year, month, latitude, longitude,
  h3_res4, h3_res5, locality, is_outlier,
  sensor_type, argos_location_class, date_id
FROM mcphersonsharyn_silver.green_turtle_silver;
-- Explicit column list ensures all four tables have matching schema
-- NULL used for argos_location_class where not applicable
-- manta ray and sperm whale use GPS not Argos

In [0]:
%sql
SELECT
  species_common,
  count(*) AS total_rows,
  count(DISTINCT animal_id) AS unique_animals,
  min(year) AS earliest_year,
  max(year) AS latest_year
FROM mcphersonsharyn_silver.migration_silver
GROUP BY species_common
ORDER BY species_common;
-- Confirms all four species present in unified view
-- Total should be 69,776 rows across 116 animals